In [1]:
from google.colab import files
uploaded = files.upload()

Saving comments.csv to comments.csv


In [2]:
import os
print(os.listdir())

['.config', 'comments.csv', 'sample_data']


In [3]:
import os
print(os.listdir())

['.config', 'comments.csv', 'sample_data']


In [4]:
import pandas as pd

df = pd.read_csv("comments.csv")  # غيّر الاسم إذا كان مختلفًا
print(df.head())
print(df.columns)
print(df.shape)

      Text
0      ✋✋✋
1     ✅✅✅✅
2  ✅✅✅✅✅✅✅
3        ⚽
4        ⛾
Index(['Text'], dtype='object')
(30000, 1)


In [5]:
import re

df = df[["Text"]].copy()
df.dropna(inplace=True)

df["Text"] = df["Text"].astype(str).str.strip()

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["Text"].apply(clean_text)

df = df[df["text"] != ""]
df = df[df["text"].str.len() > 2]
df.drop_duplicates(subset=["text"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("بعد التنظيف:", df.shape)
df.head()

بعد التنظيف: (21290, 2)


,Text,text
0,ــــــــــــــــــــــــــــــــــــــــ,ــــــــــــــــــــــــــــــــــــــــ
1,ًََ,ًََ
2,ًًً,ًًً
3,ًًًً,ًًًً
4,ًًًًً,ًًًًً


In [22]:
from datasets import load_dataset

dataset = load_dataset("ajgt_twitter_ar")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/91.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1800
    })
})
{'text': ' اربد فيها جامعات اكثر من عمان ... وفيها قد عمان ونص لعيبه المنتخب منها ... و 80 % من مطربين الاردن منها', 'label': 1}


In [23]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# تحويل البيانات إلى Dataset
train_dataset = dataset["train"]

# تقسيم Train/Test
train_test = train_dataset.train_test_split(test_size=0.2)

train_dataset = train_test["train"]
test_dataset = train_test["test"]

# تحميل AraBERT
model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# تحميل النموذج
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# التقييم
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

# إعداد التدريب
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="epoch",
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# التدريب 🔥
trainer.train()

# التقييم
results = trainer.evaluate()
print(results)

# حفظ النموذج
model.save_pretrained("final_model")
tokenizer.save_pretrained("final_model")

print("تم حفظ النموذج ✅")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.23979975283145905, 'eval_accuracy': 0.9472222222222222, 'eval_runtime': 2.7979, 'eval_samples_per_second': 128.67, 'eval_steps_per_second': 8.221, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

تم حفظ النموذج ✅


In [25]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

texts = [
    "الخدمة ممتازة جدا",
    "التطبيق سيء للغاية",
    "التجربة رائعة",
    "لا أنصح بهذا المنتج",
    "جيد لكن يحتاج تحسين"
]

for text in texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # 👇 ننقل البيانات للـ GPU
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = outputs.logits.argmax().item()

    if pred == 1:
        result = "إيجابي"
    else:
        result = "سلبي"

    print(f"{text} --> {result}")

الخدمة ممتازة جدا --> إيجابي
التطبيق سيء للغاية --> سلبي
التجربة رائعة --> إيجابي
لا أنصح بهذا المنتج --> سلبي
جيد لكن يحتاج تحسين --> إيجابي


In [26]:
import torch
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred].item()

    sentiment = "إيجابي" if pred == 1 else "سلبي"
    return f"التصنيف: {sentiment}\nالثقة: {round(confidence * 100, 2)}%"

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="اكتب تعليقًا هنا..."),
    outputs="text",
    title="Arabic Sentiment Analysis with AraBERT",
    description="اكتب نصًا عربيًا وسيقوم النموذج بتصنيفه إلى إيجابي أو سلبي مع نسبة الثقة"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e780bfd8b80da93d0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
!cp -r final_model /content/drive/MyDrive/

In [29]:
!zip -r project.zip final_model *.ipynb *.csv

	zip warning: name not matched: *.ipynb
  adding: final_model/ (stored 0%)
  adding: final_model/tokenizer_config.json (deflated 43%)
  adding: final_model/config.json (deflated 52%)
  adding: final_model/model.safetensors (deflated 7%)
  adding: final_model/tokenizer.json (deflated 74%)
  adding: comments.csv (deflated 73%)
  adding: label_comments.csv (deflated 79%)
  adding: labeled_comments.csv (deflated 79%)
  adding: sample_for_labeling.csv (deflated 79%)
